# 23 — Model Export, Dynamic Quantization & Mobile / Edge Validation

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Sections 46–48 & Section 65 (Principles 21, 31):**
> - Principle 21: *ONNX CUDA provider is explicitly verified.*
> - Principle 31: *Mobile/edge constraints are measured (actual latency, memory, model size).*
>
> **Objectives:**
> 1. Export all 4 neural models (`LIMUBERT`, `NeuralInertialOdometry`, `KalmanNetNN`, `MapGNN`) to ONNX.
> 2. Verify numerical equivalence between PyTorch and ONNX Runtime ($|y_{pt} - y_{onnx}| < 10^{-4}$).
> 3. Apply Dynamic INT8 Quantization (`.quant.onnx`) and measure compression ratio.
> 4. Benchmark per-step inference latency (p50, p95, mean ms) on CPU and GPU.
> 5. Validate end-to-end pipeline execution time against the 10 Hz smartphone budget (100 ms).

## 1. Environment & ONNX Runtime Provider Verification

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Ensure project root is on sys.path
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import onnx
import onnxruntime as ort

print('=' * 65)
print('  SIH PS 26168 — Phase 9: Model Export & Mobile Validation')
print('=' * 65)
print(f'PyTorch version     : {torch.__version__}')
print(f'CUDA Available      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device          : {torch.cuda.get_device_name(0)}')

print(f'ONNX version        : {onnx.__version__}')
print(f'ONNX Runtime version: {ort.__version__}')
providers = ort.get_available_providers()
print(f'Available Providers : {providers}')

has_cuda_provider = 'CUDAExecutionProvider' in providers
if has_cuda_provider:
    print('>>> CUDAExecutionProvider: AVAILABLE (Principle 21 verified)')
else:
    print('>>> CUDAExecutionProvider: NOT AVAILABLE (Running on CPUExecutionProvider)')

## 2. Checkpoint Inventory & Setup

In [ ]:
exports_onnx_dir = PROJECT_ROOT / 'exports' / 'onnx'
exports_quant_dir = PROJECT_ROOT / 'exports' / 'quantized'
plots_dir = PROJECT_ROOT / 'plots' / 'export'
results_dir = PROJECT_ROOT / 'results'

exports_onnx_dir.mkdir(parents=True, exist_ok=True)
exports_quant_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

models_info = {
    'limu_bert': {
        'ckpt': PROJECT_ROOT / 'checkpoints' / 'limu_bert' / 'limu_bert_best.pt',
        'onnx': exports_onnx_dir / 'limu_bert.onnx',
        'quant': exports_quant_dir / 'limu_bert.quant.onnx'
    },
    'inertial_odometry': {
        'ckpt': PROJECT_ROOT / 'checkpoints' / 'inertial_odometry' / 'inertial_odometry_best.pt',
        'onnx': exports_onnx_dir / 'inertial_odometry.onnx',
        'quant': exports_quant_dir / 'inertial_odometry.quant.onnx'
    },
    'kalmannet': {
        'ckpt': PROJECT_ROOT / 'checkpoints' / 'kalmannet' / 'kalmannet_best.pt',
        'onnx': exports_onnx_dir / 'kalmannet.onnx',
        'quant': exports_quant_dir / 'kalmannet.quant.onnx'
    },
    'map_gnn': {
        'ckpt': PROJECT_ROOT / 'checkpoints' / 'map_gnn' / 'map_gnn_best.pt',
        'onnx': exports_onnx_dir / 'map_gnn.onnx',
        'quant': exports_quant_dir / 'map_gnn.quant.onnx'
    }
}

print(f'{"Model":<22} {"Checkpoint File":<50} {"PyTorch Size":<15}')
print('-' * 90)
for name, paths in models_info.items():
    exists = paths['ckpt'].exists()
    sz_mb = paths['ckpt'].stat().st_size / (1024 * 1024) if exists else 0.0
    status = f'{sz_mb:.2f} MB' if exists else 'MISSING'
    print(f'{name:<22} {str(paths["ckpt"].relative_to(PROJECT_ROOT)):<50} {status:<15}')
    if not exists:
        raise FileNotFoundError(f'Missing required checkpoint: {paths["ckpt"]}')
print('\nAll 4 checkpoints verified.')

## 3. ONNX Model Export & Graph Validation

In [ ]:
from src.export.onnx_exporter import (
    export_limu_bert,
    export_inertial_odometry,
    export_kalmannet,
    export_map_gnn
)

export_reports = {}

# 1. Export LIMU-BERT
print('Exporting LIMU-BERT to ONNX...')
rep_lb = export_limu_bert(models_info['limu_bert']['ckpt'], models_info['limu_bert']['onnx'])
export_reports['limu_bert'] = rep_lb
print(f'  ✓ Parameters: {rep_lb["param_count"]:,} | Size: {rep_lb["size_mb"]:.2f} MB')

# 2. Export Neural Inertial Odometry
print('Exporting Neural Inertial Odometry to ONNX...')
rep_io = export_inertial_odometry(models_info['inertial_odometry']['ckpt'], models_info['inertial_odometry']['onnx'])
export_reports['inertial_odometry'] = rep_io
print(f'  ✓ Parameters: {rep_io["param_count"]:,} | Size: {rep_io["size_mb"]:.2f} MB')

# 3. Export KalmanNet
print('Exporting KalmanNet to ONNX...')
rep_kn = export_kalmannet(models_info['kalmannet']['ckpt'], models_info['kalmannet']['onnx'])
export_reports['kalmannet'] = rep_kn
print(f'  ✓ Parameters: {rep_kn["param_count"]:,} | Size: {rep_kn["size_mb"]:.2f} MB')

# 4. Export MapGNN
print('Exporting MapGNN to ONNX...')
rep_mg = export_map_gnn(models_info['map_gnn']['ckpt'], models_info['map_gnn']['onnx'])
export_reports['map_gnn'] = rep_mg
print(f'  ✓ Parameters: {rep_mg["param_count"]:,} | Size: {rep_mg["size_mb"]:.2f} MB')

# Validate ONNX Graphs
print('\nValidating ONNX Graphs via onnx.checker...')
for name, paths in models_info.items():
    model_onnx = onnx.load(str(paths['onnx']))
    onnx.checker.check_model(model_onnx)
    print(f'  ✓ {name:<22}: Graph structure VALID (opset {model_onnx.opset_import[0].version})')

## 4. Numeric Equivalence Verification (PyTorch vs ONNX Runtime)

In [ ]:
from src.export.onnx_exporter import verify_numeric_equivalence

equiv_results = {}
print(f'{"Model":<22} {"Outputs":<10} {"Max Abs Diff":<18} {"Max Rel Diff":<18} {"Status"}')
print('-' * 80)

# 1. LIMU-BERT
dummy_lb = torch.randn(2, 120, 6)
res_lb = verify_numeric_equivalence(export_reports['limu_bert']['model_ref'], models_info['limu_bert']['onnx'], dummy_lb)
equiv_results['limu_bert'] = res_lb
print(f'{"limu_bert":<22} {res_lb["num_outputs"]:^10} {res_lb["max_abs_diff"]:^18.2e} {res_lb["max_rel_diff"]:^18.2e} {"PASS ✓" if res_lb["passed"] else "FAIL ✗"}')

# 2. Neural Inertial Odometry
dummy_io = torch.randn(2, 100, 6)
res_io = verify_numeric_equivalence(export_reports['inertial_odometry']['model_ref'], models_info['inertial_odometry']['onnx'], dummy_io)
equiv_results['inertial_odometry'] = res_io
print(f'{"inertial_odometry":<22} {res_io["num_outputs"]:^10} {res_io["max_abs_diff"]:^18.2e} {res_io["max_rel_diff"]:^18.2e} {"PASS ✓" if res_io["passed"] else "FAIL ✗"}')

# 3. KalmanNet
dummy_kn = (
    torch.randn(2, 4),
    torch.randn(2, 2),
    torch.randn(2, 2),
    torch.randn(2, 2, 64)
)
res_kn = verify_numeric_equivalence(export_reports['kalmannet']['model_ref'], models_info['kalmannet']['onnx'], dummy_kn)
equiv_results['kalmannet'] = res_kn
print(f'{"kalmannet":<22} {res_kn["num_outputs"]:^10} {res_kn["max_abs_diff"]:^18.2e} {res_kn["max_rel_diff"]:^18.2e} {"PASS ✓" if res_kn["passed"] else "FAIL ✗"}')

# 4. MapGNN
dummy_mg = (
    torch.randn(2, 6),
    torch.randn(2, 5, 6),
    torch.eye(5).unsqueeze(0).repeat(2, 1, 1)
)
res_mg = verify_numeric_equivalence(export_reports['map_gnn']['model_ref'], models_info['map_gnn']['onnx'], dummy_mg)
equiv_results['map_gnn'] = res_mg
print(f'{"map_gnn":<22} {res_mg["num_outputs"]:^10} {res_mg["max_abs_diff"]:^18.2e} {res_mg["max_rel_diff"]:^18.2e} {"PASS ✓" if res_mg["passed"] else "FAIL ✗"}')

all_passed = all(r['passed'] for r in equiv_results.values())
if not all_passed:
    raise ValueError('One or more models failed numerical equivalence verification!')
print('\nAll 4 models successfully validated for numerical equivalence (< 1e-4).')

## 5. Dynamic INT8 Quantization (`.quant.onnx`)

In [ ]:
from src.export.onnx_exporter import quantize_onnx_model, profile_model_footprint

quant_reports = {}
print(f'{"Model":<22} {"FP32 ONNX":<15} {"INT8 ONNX":<15} {"Reduction":<15} {"Ratio":<10}')
print('-' * 80)

for name, paths in models_info.items():
    q_rep = quantize_onnx_model(paths['onnx'], paths['quant'])
    quant_reports[name] = q_rep
    print(f'{name:<22} {q_rep["original_mb"]:>7.2f} MB   {q_rep["quantized_mb"]:>7.2f} MB   {q_rep["size_reduction_pct"]:>6.1f}%        {q_rep["compression_ratio"]:>5.2f}x')

total_fp32 = sum(r['original_mb'] for r in quant_reports.values())
total_int8 = sum(r['quantized_mb'] for r in quant_reports.values())
total_savings = (1.0 - total_int8 / total_fp32) * 100.0
print('-' * 80)
print(f'{"TOTAL SUITE":<22} {total_fp32:>7.2f} MB   {total_int8:>7.2f} MB   {total_savings:>6.1f}%        {total_fp32/total_int8:>5.2f}x')

## 6. Real-Time Latency Benchmarking (CPU & GPU Inference Profile)

In [ ]:
from src.export.onnx_exporter import benchmark_latency

benchmark_results = {}
sample_inputs = {
    'limu_bert': torch.randn(1, 120, 6),
    'inertial_odometry': torch.randn(1, 100, 6),
    'kalmannet': (torch.randn(1, 4), torch.randn(1, 2), torch.randn(1, 2), torch.zeros(2, 1, 64)),
    'map_gnn': (torch.randn(1, 6), torch.randn(1, 5, 6), torch.eye(5).unsqueeze(0))
}

print(f'{"Model":<20} {"Backend":<15} {"Mean (ms)":<12} {"P50 (ms)":<12} {"P95 (ms)":<12} {"FPS":<10}')
print('-' * 85)

for name in models_info.keys():
    benchmark_results[name] = {}
    inp = sample_inputs[name]
    
    # 1. PyTorch CPU
    pt_model = export_reports[name]['model_ref'].to('cpu')
    b_pt = benchmark_latency(pt_model, inp, is_onnx=False, n_runs=100, warmup=20)
    benchmark_results[name]['pytorch_cpu'] = b_pt
    print(f'{name:<20} {"PyTorch CPU":<15} {b_pt["mean_ms"]:>8.2f} ms {b_pt["p50_ms"]:>8.2f} ms {b_pt["p95_ms"]:>8.2f} ms {b_pt["throughput_fps"]:>8.1f}')
    
    # 2. ONNX Runtime CPU (FP32)
    sess_cpu = ort.InferenceSession(str(models_info[name]['onnx']), providers=['CPUExecutionProvider'])
    b_ort_cpu = benchmark_latency(sess_cpu, inp, is_onnx=True, n_runs=100, warmup=20)
    benchmark_results[name]['onnx_cpu_fp32'] = b_ort_cpu
    print(f'{name:<20} {"ONNX CPU (FP32)":<15} {b_ort_cpu["mean_ms"]:>8.2f} ms {b_ort_cpu["p50_ms"]:>8.2f} ms {b_ort_cpu["p95_ms"]:>8.2f} ms {b_ort_cpu["throughput_fps"]:>8.1f}')
    
    # 3. ONNX Runtime CPU (INT8 Quantized)
    sess_quant = ort.InferenceSession(str(models_info[name]['quant']), providers=['CPUExecutionProvider'])
    b_quant = benchmark_latency(sess_quant, inp, is_onnx=True, n_runs=100, warmup=20)
    benchmark_results[name]['onnx_cpu_int8'] = b_quant
    print(f'{name:<20} {"ONNX CPU (INT8)":<15} {b_quant["mean_ms"]:>8.2f} ms {b_quant["p50_ms"]:>8.2f} ms {b_quant["p95_ms"]:>8.2f} ms {b_quant["throughput_fps"]:>8.1f}')
    
    # 4. ONNX Runtime GPU (if CUDA available)
    if has_cuda_provider:
        sess_gpu = ort.InferenceSession(str(models_info[name]['onnx']), providers=['CUDAExecutionProvider'])
        b_gpu = benchmark_latency(sess_gpu, inp, is_onnx=True, n_runs=100, warmup=20)
        benchmark_results[name]['onnx_gpu_cuda'] = b_gpu
        print(f'{name:<20} {"ONNX GPU (CUDA)":<15} {b_gpu["mean_ms"]:>8.2f} ms {b_gpu["p50_ms"]:>8.2f} ms {b_gpu["p95_ms"]:>8.2f} ms {b_gpu["throughput_fps"]:>8.1f}')
    
    print('-' * 85)

## 7. End-to-End Smartphone 10 Hz Real-Time Budget Analysis

In [ ]:
# Real-time 10 Hz navigation requires step execution under 100 ms.
# In the online navigation loop, each 10 Hz step executes:
#   - Invariant ESKF propagation + NHC update: ~0.15 ms
#   - Neural Inertial Odometry forward inference: t_io
#   - KalmanNet adaptive gain update: t_kn
#   - Topological MapGNN candidate ranking: t_mg

eskf_nhc_latency_ms = 0.15
t_io_cpu = benchmark_results['inertial_odometry']['onnx_cpu_int8']['p50_ms']
t_kn_cpu = benchmark_results['kalmannet']['onnx_cpu_int8']['p50_ms']
t_mg_cpu = benchmark_results['map_gnn']['onnx_cpu_int8']['p50_ms']

total_step_latency_cpu = eskf_nhc_latency_ms + t_io_cpu + t_kn_cpu + t_mg_cpu
budget_ms = 100.0  # 10 Hz
cpu_headroom_pct = ((budget_ms - total_step_latency_cpu) / budget_ms) * 100.0

print('=' * 65)
print('  SIH PS 26168 — 10 Hz Real-Time Smartphone Budget Analysis')
print('=' * 65)
print(f'1. ESKF + NHC Kinematic Propagation : {eskf_nhc_latency_ms:>6.2f} ms')
print(f'2. Neural Inertial Odometry (INT8)  : {t_io_cpu:>6.2f} ms')
print(f'3. KalmanNet Adaptive Gain (INT8)    : {t_kn_cpu:>6.2f} ms')
print(f'4. MapGNN Topology Matching (INT8)   : {t_mg_cpu:>6.2f} ms')
print('-' * 65)
print(f'TOTAL ESTIMATED STEP TIME (CPU)      : {total_step_latency_cpu:>6.2f} ms')
print(f'10 Hz Real-Time Budget              : {budget_ms:>6.2f} ms')
print(f'Available CPU Idle Headroom         : {cpu_headroom_pct:>6.1f} %')
print(f'Verdict                             : {"REAL-TIME CAPABLE ✓" if total_step_latency_cpu < budget_ms else "OVER BUDGET ✗"}')

## 8. Diagnostic Visualizations & Reporting

In [ ]:
# 1. Plot Latency Comparison
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
models = list(models_info.keys())
x = np.arange(len(models))
width = 0.22

pt_lat = [benchmark_results[m]['pytorch_cpu']['p50_ms'] for m in models]
ort_fp32_lat = [benchmark_results[m]['onnx_cpu_fp32']['p50_ms'] for m in models]
ort_int8_lat = [benchmark_results[m]['onnx_cpu_int8']['p50_ms'] for m in models]

rects1 = ax.bar(x - width, pt_lat, width, label='PyTorch CPU (FP32)', color='#e05d44')
rects2 = ax.bar(x, ort_fp32_lat, width, label='ONNX Runtime CPU (FP32)', color='#007ec6')
rects3 = ax.bar(x + width, ort_int8_lat, width, label='ONNX Runtime CPU (INT8)', color='#44cc11')

ax.set_ylabel('Inference Latency P50 (ms)')
ax.set_title('SIH PS 26168 — Model Inference Latency by Runtime Backend')
ax.set_xticks(x)
ax.set_xticklabels(['LIMU-BERT', 'Inertial Odo', 'KalmanNet', 'MapGNN'], fontweight='bold')
ax.axhline(100.0, color='r', linestyle='--', alpha=0.5, label='10 Hz Step Budget (100 ms)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

for rects in [rects1, rects2, rects3]:
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
latency_fig_path = plots_dir / 'model_latency_comparison.png'
plt.savefig(latency_fig_path)
plt.close()
print(f'Saved: {latency_fig_path}')

# 2. Plot Footprint Compression
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
pt_sz = [models_info[m]['ckpt'].stat().st_size / (1024 * 1024) for m in models]
fp32_sz = [quant_reports[m]['original_mb'] for m in models]
int8_sz = [quant_reports[m]['quantized_mb'] for m in models]

rects1 = ax.bar(x - width, pt_sz, width, label='PyTorch Checkpoint (.pt)', color='#9b59b6')
rects2 = ax.bar(x, fp32_sz, width, label='ONNX Model (FP32)', color='#3498db')
rects3 = ax.bar(x + width, int8_sz, width, label='Quantized ONNX (INT8)', color='#2ecc71')

ax.set_ylabel('Model Disk Size (MB)')
ax.set_title('SIH PS 26168 — Model Footprint & Dynamic INT8 Compression')
ax.set_xticks(x)
ax.set_xticklabels(['LIMU-BERT', 'Inertial Odo', 'KalmanNet', 'MapGNN'], fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

for rects in [rects1, rects2, rects3]:
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:.2f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
footprint_fig_path = plots_dir / 'model_footprint_compression.png'
plt.savefig(footprint_fig_path)
plt.close()
print(f'Saved: {footprint_fig_path}')

## 9. Serialize Final Export Metrics JSON

In [ ]:
export_summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()),
    'onnx_runtime_version': ort.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_provider_available': has_cuda_provider,
    'models': {},
    'total_footprint': {
        'pytorch_total_mb': sum(pt_sz),
        'onnx_fp32_total_mb': total_fp32,
        'onnx_int8_total_mb': total_int8,
        'overall_savings_pct': total_savings,
        'overall_compression_ratio': total_fp32 / max(total_int8, 1e-6)
    },
    'mobile_realtime_budget': {
        'target_frequency_hz': 10,
        'budget_per_step_ms': budget_ms,
        'estimated_step_latency_cpu_ms': total_step_latency_cpu,
        'available_cpu_headroom_pct': cpu_headroom_pct,
        'is_realtime_compliant': bool(total_step_latency_cpu < budget_ms)
    }
}

for name in models_info.keys():
    export_summary['models'][name] = {
        'param_count': export_reports[name]['param_count'],
        'numeric_equivalence': {
            'passed': equiv_results[name]['passed'],
            'max_abs_diff': equiv_results[name]['max_abs_diff'],
            'max_rel_diff': equiv_results[name]['max_rel_diff']
        },
        'footprint': quant_reports[name],
        'latency': benchmark_results[name]
    }

metrics_path = results_dir / 'model_export_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(export_summary, f, indent=2)

print(f'Export metrics written to: {metrics_path}')
print('\n' + '=' * 65)
print('  PHASE 9: MODEL EXPORT & MOBILE VALIDATION COMPLETE')
print('=' * 65)